# Personal Commute Time Prediction
**Route:** Chethicode → Knovista Technologies, Kadavanthra Junction, Kochi.

**Data warning:** The Excel workbook contains 100 simulated trips, not recovered real journeys or verified historical weather/traffic. This notebook is a prototype. Do not claim its model scores represent real-world accuracy.

**Target:** `Travel_Duration_Min`. `Arrival_Time` is deliberately excluded from model inputs to prevent target leakage.

## 1. Upload and load the Excel dataset

In [ ]:
# In Google Colab:
from google.colab import files
uploaded = files.upload()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

file_name = next(iter(uploaded))
df = pd.read_excel(file_name, sheet_name="Commute_Data")
print(df.shape)
display(df.head())
print(df.dtypes)

## 2. Clean and inspect

In [ ]:
data = df.copy()
data.columns = data.columns.str.strip()
for col in ["Weather", "Traffic"]:
    data[col] = data[col].astype("string").str.strip().str.title()

data["Date"] = pd.to_datetime(data["Date"], errors="coerce")
data["Departure_Time"] = pd.to_datetime(
    data["Departure_Time"].astype(str), errors="coerce
).dt.time

data["Travel_Duration_Min"] = pd.to_numeric(
    data["Travel_Duration_Min"], errors="coerce"
)
data = data.drop_duplicates(subset=["Trip_ID"])
data = data.dropna(subset=[
    "Date", "Departure_Time", "Weather", "Traffic", "Travel_Duration_Min"
])
data = data[data["Travel_Duration_Min"].between(1, 240)]
data = data.sort_values(["Date", "Departure_Time"]).reset_index(drop=True)

print("Cleaned shape:", data.shape)
print("Date range:", data["Date"].min(), "to", data["Date"].max())
print("Missing values:\n", data.isna().sum())
display(data.head())

## 3. Exploratory data analysis

In [ ]:
display(data["Travel_Duration_Min"].describe())
print(data["Weather"].value_counts())
print(data["Traffic"].value_counts())

plt.figure(figsize=(8,4))
sns.histplot(data["Travel_Duration_Min"], bins=15, kde=True)
plt.title("Commute duration distribution (simulated)")
plt.xlabel("Minutes")
plt.show()

plt.figure(figsize=(8,4))
sns.boxplot(data=data, x="Traffic", y="Travel_Duration_Min",
            order=["Low", "Moderate", "Heavy"])
plt.title("Duration by traffic category (simulated)")
plt.show()

plt.figure(figsize=(8,4))
sns.boxplot(data=data, x="Weather", y="Travel_Duration_Min",
            order=["Clear", "Cloudy", "Rainy"])
plt.title("Duration by weather category (simulated)")
plt.show()

daily = data.groupby("Date", as_index=False)["Travel_Duration_Min"].mean()
plt.figure(figsize=(10,4))
plt.plot(daily["Date"], daily["Travel_Duration_Min"], marker=".")
plt.title("Commute duration over time (simulated)")
plt.ylabel("Minutes")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 4. Feature engineering

In [ ]:
def add_features(frame):
    out = frame.copy()
    out["departure_minutes"] = out["Departure_Time"].apply(
        lambda t: t.hour * 60 + t.minute
    )
    out["departure_hour"] = out["Departure_Time"].apply(lambda t: t.hour)
    out["departure_minute"] = out["Departure_Time"].apply(lambda t: t.minute)
    out["day_of_week"] = out["Date"].dt.dayofweek
    out["month"] = out["Date"].dt.month
    out["is_saturday"] = (out["day_of_week"] == 5).astype(int)
    out["is_morning_peak"] = out["departure_minutes"].between(
        7*60+45, 8*60+45
    ).astype(int)
    return out

featured = add_features(data)
numeric_features = [
    "departure_minutes", "departure_hour", "departure_minute",
    "day_of_week", "month", "is_saturday", "is_morning_peak"
]
categorical_features = ["Weather", "Traffic"]
feature_columns = numeric_features + categorical_features
X = featured[feature_columns]
y = featured["Travel_Duration_Min"]

## 5. Chronological train/test split
The earliest 80% is used for training and the latest 20% for testing.

In [ ]:
split_at = int(len(featured) * 0.80)
X_train, X_test = X.iloc[:split_at], X.iloc[split_at:]
y_train, y_test = y.iloc[:split_at], y.iloc[split_at:]
print("Train rows:", len(X_train), "| Test rows:", len(X_test))
print("Train period:", featured.iloc[0]["Date"], "to", featured.iloc[split_at-1]["Date"])
print("Test period:", featured.iloc[split_at]["Date"], "to", featured.iloc[-1]["Date"])

## 6. Train and compare regression models

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

numeric_prep = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
category_prep = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_prep, numeric_features),
    ("cat", category_prep, categorical_features)
])

estimators = {
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(
        n_estimators=250, max_depth=5, min_samples_leaf=2, random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100, learning_rate=0.04, max_depth=2, random_state=42
    )
}
trained = {}
rows = []

for name, estimator in estimators.items():
    pipe = Pipeline([("prep", preprocessor), ("model", estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    trained[name] = pipe
    rows.append({
        "Model": name,
        "MAE_minutes": mean_absolute_error(y_test, pred),
        "RMSE_minutes": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred)
    })

results = pd.DataFrame(rows).sort_values("MAE_minutes")
display(results)

## 7. Evaluate the best model
With only 100 synthetic rows, holdout scores are illustrative and can be unstable.

In [ ]:
best_name = results.iloc[0]["Model"]
best_model = trained[best_name]
pred = best_model.predict(X_test)

print("Selected model:", best_name)
print("MAE:", round(mean_absolute_error(y_test, pred), 2), "minutes")
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, pred)), 2), "minutes")
print("R²:", round(r2_score(y_test, pred), 3))

plt.figure(figsize=(6,6))
plt.scatter(y_test, pred)
lo = min(y_test.min(), pred.min())
hi = max(y_test.max(), pred.max())
plt.plot([lo, hi], [lo, hi], "--")
plt.xlabel("Actual minutes")
plt.ylabel("Predicted minutes")
plt.title("Actual vs predicted — synthetic test set")
plt.show()

## 8. Save the trained model

In [ ]:
import joblib
artifact = {
    "model": best_model,
    "model_name": best_name,
    "feature_columns": feature_columns,
    "data_notice": "Trained on synthetic illustrative data; not validated for real-world use."
}
joblib.dump(artifact, "commute_time_model.joblib")
results.to_csv("model_comparison.csv", index=False)

from google.colab import files
files.download("commute_time_model.joblib")
files.download("model_comparison.csv")

## 9. Try a prediction

In [ ]:
from datetime import time

def predict_duration(travel_date, departure_time, weather, traffic):
    sample = pd.DataFrame([{
        "Date": pd.Timestamp(travel_date),
        "Departure_Time": departure_time,
        "Weather": weather,
        "Traffic": traffic
    }])
    sample = add_features(sample)
    minutes = best_model.predict(sample[feature_columns])[0]
    return round(max(1, float(minutes)), 1)

# Example only; change the values:
estimate = predict_duration(
    "2026-11-09", time(8, 10), "Rainy", "Heavy"
)
print("Prototype estimate:", estimate, "minutes")

## 10. Portfolio and validation checklist

- Replace simulated trips with actual logged observations when available.
- Use real recorded weather/traffic categories, not guessed historical labels.
- Collect more trips across seasons and departure-time ranges.
- Validate on a later real-world period before claiming model accuracy.
- Do not present synthetic model metrics as actual commute performance.

Suggested portfolio description: **Built a commute-time prediction prototype using a clearly labelled synthetic dataset, including data cleaning, EDA, feature engineering, chronological evaluation and regression model comparison.**